# Topological feature groups

Topology-only models use all descriptors or omit one group, on the same 25 folds.

The saved tables can be viewed without rerunning the models.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../../data/README.md) and
[DataPreparation.ipynb](../../../data/DataPreparation.ipynb), you can rerun the
ablation and regenerate the tables using the functions in
[`src/dtpkg/ablation/`](../../../src/dtpkg/ablation/).

## Paths

Fill in `PROJECT_ROOT` to read the saved tables. The remaining placeholders are
needed for training; `REFERENCE_DIR` is the completed [transductive run](../../04_transductive_fusion/README.md).

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = ...  # DTP-KG checkout
DATA_DIR = ...  # Prepared private data
GRAPH_PATH = ...  # Canonical GraphML network
REFERENCE_DIR = ...  # Completed transductive run from experiment 04

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

EXPERIMENT = PROJECT_ROOT / "experiments" / "06_ablation" / "topology_feature_groups"
TABLES = EXPERIMENT / "tables"
RESULTS_ROOT = EXPERIMENT / "results" / "runs"
OUTPUT_DIR = EXPERIMENT / "results" / "table_exports"
RUN_TRAINING = False
STUDY = "topological_features"

## Saved results

The tables use fold means and sample SD. Paired differences use approximate corrected-CV intervals.

In [ ]:
from IPython.display import display
from dtpkg.ablation.public_tables import load_report, performance_table, contrast_table

report = load_report(TABLES, study=STUDY)
display(performance_table(report))
display(contrast_table(report))

,Features,AUROC,F1
0,All descriptors,0.9450 ± 0.0059,0.8674 ± 0.0094
1,Without degree/size,0.8671 ± 0.0095,0.7945 ± 0.0103
2,Without clustering/boundary,0.9397 ± 0.0066,0.8574 ± 0.0098
3,Without centrality,0.9475 ± 0.0060,0.8727 ± 0.0082


,Contrast,ΔAUROC,ΔF1
0,All − without degree/size,+0.07798**,+0.07283**
1,All − without clustering/boundary,+0.00537,+0.01002
2,All − without centrality,-0.00249,-0.00529


Degree/size includes the seven degree descriptors and neighbourhood size;
clustering/boundary contains local clustering and boundary ratio; centrality
contains local closeness and Katz. The six AUROC/F1 contrasts share one Holm family.

## Regenerate the results

Complete the [transductive experiment](../../04_transductive_fusion/TransductiveFusion.ipynb)
and set `REFERENCE_DIR` to its run directory, then enable `RUN_TRAINING`.
The run reuses compatible transductive fits and trains the remaining arms.

In [ ]:
if RUN_TRAINING:
    from dtpkg.ablation.workflow import prepare_study
    from dtpkg.ablation.public_tables import export_tables

    study = prepare_study(
        study=STUDY,
        data_dir=DATA_DIR,
        graph_path=GRAPH_PATH,
        reference_dir=REFERENCE_DIR,
        results_root=RESULTS_ROOT,
    )
    study.run()
    export_tables(study.output_dir, OUTPUT_DIR, study=STUDY)
    new_report = load_report(OUTPUT_DIR, study=STUDY)
    display(performance_table(new_report))
    display(contrast_table(new_report))